# 05 — Deep Learning e Transformers

Implementa a Fase 18 do plano de elaboração: treina LSTM, CNN, BERTimbau,
RoBERTa e DistilBERT (`configs/model_params.yaml -> deep_learning`/
`transformers`) diretamente sobre o texto cru — ao contrário dos modelos
clássicos (`notebooks/04_ml_classico.ipynb`), estes não dependem de uma
matriz TF-IDF ajustada ao treino, então `paths.test_corpus_file` pode ser
usado diretamente como conjunto de teste real.

**Aviso de custo computacional**: treina cinco modelos, três deles
Transformers — considere uma GPU disponível e/ou reduzir
`configs/model_params.yaml -> transformers.*.epochs` para uma execução
exploratória rápida.

**Pré-requisito**: a etapa `labeling` já deve ter sido executada.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

from config.constants import CONFIG_FILE_NAMES
from config.paths import CONFIGS_DIR, load_project_paths
from data.loader import load_training_example_dataset
from evaluation.evaluator import evaluate_classifier
from evaluation.reports import (
    build_evaluation_report,
    merge_evaluation_reports,
    save_evaluation_report,
)
from io_utils.yaml import read_yaml
from pipelines.training_deep_learning import (
    DEFAULT_DEEP_LEARNING_MODEL_NAMES,
    run_training_deep_learning_stage,
)
from visualization.confusion_matrix import plot_confusion_matrix_heatmap
from visualization.roc_pr_curves import plot_roc_curves_one_vs_rest
from visualization.theme import apply_project_theme, save_figure

apply_project_theme()
paths = load_project_paths()

training_corpus = load_training_example_dataset(paths.training_corpus_file)
test_corpus = load_training_example_dataset(paths.test_corpus_file)
X_train, y_train = training_corpus["text"].to_list(), training_corpus["sentiment_label"].to_list()
X_test, y_test = test_corpus["text"].to_list(), test_corpus["sentiment_label"].to_list()

## Treino dos cinco modelos

Reaproveita `pipelines.training_deep_learning.run_training_deep_learning_stage`
diretamente — mesma função usada por `src/main.py --stage
training_deep_learning` — mapeando os hiperparâmetros de cada modelo a
partir de `configs/model_params.yaml`, exatamente como em `src/main.py`.

In [ ]:
_MODEL_PARAM_KEYS = {
    "lstm": ("deep_learning", "recurrent"),
    "cnn": ("deep_learning", "convolutional"),
    "bertimbau": ("transformers", "bertimbau"),
    "roberta": ("transformers", "roberta"),
    "distilbert": ("transformers", "distilbert"),
}
model_params_by_section = read_yaml(CONFIGS_DIR / CONFIG_FILE_NAMES["model_params"])
model_params = {
    model_name: model_params_by_section[section][subsection]
    for model_name, (section, subsection) in _MODEL_PARAM_KEYS.items()
}

training_results = run_training_deep_learning_stage(
    X_train,
    y_train,
    None,
    None,
    model_names=DEFAULT_DEEP_LEARNING_MODEL_NAMES,
    model_params=model_params,
    checkpoints_dir=paths.models_checkpoints_dir,
    track_with_mlflow=True,
)

## Avaliação sobre o conjunto de teste

In [ ]:
evaluation_reports = []
for model_name, training_result in training_results.items():
    model = training_result.model
    y_pred = model.predict(X_test)
    y_score = model.predict_proba(X_test)

    evaluation_result = evaluate_classifier(y_test, y_pred, y_score=y_score)
    print(f"{model_name}: {evaluation_result.point_metrics}")

    confusion_figure = plot_confusion_matrix_heatmap(
        evaluation_result.confusion_matrix, title=f"Matriz de Confusão — {model_name}"
    )
    save_figure(
        confusion_figure, f"matriz_confusao_{model_name}", directory=paths.reports_figures_dir
    )

    roc_figure = plot_roc_curves_one_vs_rest(y_test, y_score, title=f"Curvas ROC — {model_name}")
    save_figure(roc_figure, f"curvas_roc_{model_name}", directory=paths.reports_figures_dir)

    evaluation_reports.append(build_evaluation_report(model_name, evaluation_result))

deep_learning_report = merge_evaluation_reports(evaluation_reports)
save_evaluation_report(
    deep_learning_report, paths.reports_metrics_dir / "avaliacao_deep_learning_transformers.csv"
)
deep_learning_report

## Conclusões

Registrar aqui: (1) se os Transformers (BERTimbau/RoBERTa/DistilBERT)
superam LSTM/CNN por margem que justifique seu custo computacional bem
maior — comparação formal de significância fica em
`notebooks/07_avaliacao_comparativa.ipynb`; (2) sinais de overfitting
(early stopping disparando muito cedo/tarde — ver
`configs/model_params.yaml -> *.early_stopping_patience`); (3) tempo de
treino e uso de memória por modelo, relevantes para a escolha final de
deploy (`configs/deploy.yaml -> api.default_approach`).